In [2]:
import pandas as pd

train = pd.read_csv("../data/fraudTrain.csv")
test = pd.read_csv("../data/fraudTest.csv")


for df in (train,test):
    df["time"] = pd.to_datetime(df["trans_date_trans_time"])


train = train.sort_values("time").reset_index(drop = True) 
test = test.sort_values("time").reset_index(drop = True) 


print(train["time"].is_monotonic_increasing, test["time"].is_monotonic_increasing)
print(len(train), len(test)) 

True True
1296675 555719


In [3]:
for df in (train,test):
    df["hour"] = df["time"].dt.hour
    df["is_night"] = df["hour"].isin([22,23,0,1,2,3]).astype(int)


print(train.groupby("is_night")["is_fraud"].mean()*100)

is_night
0    0.115347
1    2.086709
Name: is_fraud, dtype: float64


In [4]:
categories = sorted(train["category"].unique()) 
print(len(categories), categories)

for df in (train , test):
    for c in categories:
        df["cat_" + c] = (df["category"] == c).astype(int) 

print(train[["category", "cat_shopping_net", "cat_home"]].head(5)) 
    
    

14 ['entertainment', 'food_dining', 'gas_transport', 'grocery_net', 'grocery_pos', 'health_fitness', 'home', 'kids_pets', 'misc_net', 'misc_pos', 'personal_care', 'shopping_net', 'shopping_pos', 'travel']
        category  cat_shopping_net  cat_home
0       misc_net                 0         0
1    grocery_pos                 0         0
2  entertainment                 0         0
3  gas_transport                 0         0
4       misc_pos                 0         0


In [5]:
all_df = pd.concat([train.assign(part = "train"), test.assign(part = "test")], ignore_index = True)
all_df = all_df.sort_values("time", kind = "stable").reset_index(drop = True)

g = all_df.groupby("cc_num")["amt"]
all_df["prev_count"] = g.cumcount()
all_df["card_avg_amt"] = (g.cumsum() - all_df["amt"]) / all_df["prev_count"]
all_df["amt_vs_usual"] = all_df["amt"] / all_df["card_avg_amt"]

print(all_df[["cc_num", "amt", "prev_count", "card_avg_amt", "amt_vs_usual"]].head(8))
print(all_df.groupby("is_fraud")["amt_vs_usual"].median())

             cc_num     amt  prev_count  card_avg_amt  amt_vs_usual
0  2703186189652095    4.97           0           NaN           NaN
1      630423337322  107.23           0           NaN           NaN
2    38859492057661  220.11           0           NaN           NaN
3  3534093764340240   45.00           0           NaN           NaN
4   375534208663984   41.96           0           NaN           NaN
5  4767265376804500   94.63           0           NaN           NaN
6    30074693890476   44.54           0           NaN           NaN
7  6011360759745864   71.65           0           NaN           NaN
is_fraud
0    0.664783
1    5.295197
Name: amt_vs_usual, dtype: float64


In [6]:
import numpy as np

def recent_count(times, window_minutes):
    t = times.values.astype("datetime64[ns]")
    start = np.searchsorted(t, t - np.timedelta64(window_minutes, "m"), side="left")
    return np.arange(len(t)) - start

recent_10 = np.zeros(len(all_df), dtype=int)
recent_60 = np.zeros(len(all_df), dtype=int)

for card, idx in all_df.groupby("cc_num").groups.items():
    times = all_df.loc[idx, "time"]
    recent_10[idx] = recent_count(times, 10)
    recent_60[idx] = recent_count(times, 60)

all_df["recent_10min"] = recent_10
all_df["recent_60min"] = recent_60

print(all_df.groupby("is_fraud")[["recent_10min", "recent_60min"]].mean())

          recent_10min  recent_60min
is_fraud                            
0             0.033327      0.197525
1             0.132629      0.676925


In [7]:
all_df["prev_time"] = all_df.groupby("cc_num")["time"].shift(1)
all_df["mins_since_last"] = (all_df["time"] - all_df["prev_time"]).dt.total_seconds() / 60

print(all_df.groupby("is_fraud")["mins_since_last"].median())

is_fraud
0    262.383333
1     81.916667
Name: mins_since_last, dtype: float64


In [8]:
all_df["seen_before"] = all_df.groupby(["cc_num", "merchant"]).cumcount()
all_df["is_new_merchant"] = (all_df["seen_before"] == 0).astype(int)

print(all_df.groupby("is_fraud")["is_new_merchant"].mean() * 100)

is_fraud
0    28.485253
1    45.777640
Name: is_new_merchant, dtype: float64


In [9]:
train_part = all_df[all_df["part"] == "train"]

fill_values = {
    "card_avg_amt": train_part["amt"].median(),
    "amt_vs_usual": 1.0,
    "mins_since_last": train_part["mins_since_last"].median(),
}

all_df["has_history"] = (all_df["prev_count"] > 0).astype(int)

for col, val in fill_values.items():
    all_df[col] = all_df[col].fillna(val)

print(fill_values) 
print(all_df[["has_history", "card_avg_amt", "amt_vs_usual", "mins_since_last"]].isna().sum())

{'card_avg_amt': np.float64(47.52), 'amt_vs_usual': 1.0, 'mins_since_last': np.float64(276.0)}
has_history        0
card_avg_amt       0
amt_vs_usual       0
mins_since_last    0
dtype: int64


In [10]:
check = all_df.sample(1000, random_state=1)
groups = all_df.groupby("cc_num").indices

bad = 0
for i, row in check.iterrows():
    pos = groups[row["cc_num"]]
    hist = all_df.iloc[pos[pos < i]]
    t = row["time"]

    ok = row["prev_count"] == len(hist)
    ok &= row["recent_10min"] == (hist["time"] >= t - pd.Timedelta(minutes=10)).sum()
    ok &= row["recent_60min"] == (hist["time"] >= t - pd.Timedelta(minutes=60)).sum()
    ok &= row["is_new_merchant"] == int((hist["merchant"] == row["merchant"]).sum() == 0)
    if len(hist) > 0:
        ok &= abs(row["card_avg_amt"] - hist["amt"].mean()) < 1e-6
        ok &= abs(row["mins_since_last"] - (t - hist["time"].iloc[-1]).total_seconds() / 60) < 1e-6
    if not ok:
        bad += 1

print("rows checked:", len(check), "| rows that did not match:", bad)

rows checked: 1000 | rows that did not match: 0


In [11]:
%%writefile ../src/features.py
"""Builds the model's input numbers (features) for one payment at a time.

The same code is used for training and for the live checker,
so both see identical numbers.
"""
from collections import deque
from datetime import timedelta

CATEGORIES = [
    "entertainment", "food_dining", "gas_transport", "grocery_net", "grocery_pos",
    "health_fitness", "home", "kids_pets", "misc_net", "misc_pos",
    "personal_care", "shopping_net", "shopping_pos", "travel",
]
NIGHT_HOURS = {22, 23, 0, 1, 2, 3}

# Values used when a card has no earlier payments (worked out from the train data)
FILL_CARD_AVG_AMT = 47.52
FILL_AMT_VS_USUAL = 1.0
FILL_MINS_SINCE_LAST = 276.0

FEATURE_COLUMNS = (
    ["amt", "hour", "is_night"]
    + ["cat_" + c for c in CATEGORIES]
    + ["card_avg_amt", "amt_vs_usual", "recent_10min", "recent_60min",
       "mins_since_last", "is_new_merchant", "has_history"]
)


class CardHistory:
    """What we remember about one card's earlier payments."""

    def __init__(self):
        self.count = 0
        self.amt_sum = 0.0
        self.last_time = None
        self.merchants = set()
        self.recent_times = deque()  # times of payments in the last 60 minutes


class FeatureBuilder:
    """Keeps a memory of every card and turns each new payment into features."""

    def __init__(self):
        self.cards = {}

    def process(self, payment):
        """payment: dict with cc_num, merchant, category, amt and time (a Timestamp).
        Returns a dict of features, then adds the payment to the card's memory."""
        t = payment["time"]
        h = self.cards.setdefault(payment["cc_num"], CardHistory())

        # forget payments older than 60 minutes
        while h.recent_times and h.recent_times[0] < t - timedelta(minutes=60):
            h.recent_times.popleft()

        has_history = h.count > 0
        card_avg = h.amt_sum / h.count if has_history else FILL_CARD_AVG_AMT
        amt_vs_usual = payment["amt"] / card_avg if has_history else FILL_AMT_VS_USUAL
        mins_since_last = (
            (t - h.last_time).total_seconds() / 60 if has_history else FILL_MINS_SINCE_LAST
        )

        f = {
            "amt": payment["amt"],
            "hour": t.hour,
            "is_night": int(t.hour in NIGHT_HOURS),
            "card_avg_amt": card_avg,
            "amt_vs_usual": amt_vs_usual,
            "recent_10min": sum(1 for x in h.recent_times if x >= t - timedelta(minutes=10)),
            "recent_60min": len(h.recent_times),
            "mins_since_last": mins_since_last,
            "is_new_merchant": int(payment["merchant"] not in h.merchants),
            "has_history": int(has_history),
        }
        for c in CATEGORIES:
            f["cat_" + c] = int(payment["category"] == c)

        # only now remember this payment, so it never counts as its own history
        h.count += 1
        h.amt_sum += payment["amt"]
        h.last_time = t
        h.merchants.add(payment["merchant"])
        h.recent_times.append(t)
        return f

Overwriting ../src/features.py


In [12]:
import sys
import numpy as np
sys.path.append("../src")
from features import FeatureBuilder, FEATURE_COLUMNS

N = 200000
sub = all_df.iloc[:N]

fb = FeatureBuilder()
rows = []
for cc, m, cat, amt, t in zip(sub["cc_num"], sub["merchant"], sub["category"], sub["amt"], sub["time"]):
    rows.append(fb.process({"cc_num": cc, "merchant": m, "category": cat, "amt": amt, "time": t}))

live = pd.DataFrame(rows)[FEATURE_COLUMNS]
table = sub[FEATURE_COLUMNS].reset_index(drop=True)

differ = {}
for col in FEATURE_COLUMNS:
    same = np.isclose(live[col].astype(float), table[col].astype(float), rtol=1e-6, atol=1e-6)
    if not same.all():
        differ[col] = int((~same).sum())

print("payments compared:", N)
print("columns that differ:", differ if differ else "none")

payments compared: 200000
columns that differ: none


## **Save the feature table**

In [13]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os

cols = ["trans_num", "cc_num", "time", "part", "is_fraud"] + FEATURE_COLUMNS
all_df[cols].to_parquet("../data/features.parquet", index=False)

print(all_df[cols].shape)
print(round(os.path.getsize("../data/features.parquet") / 1e6), "MB")

(1852394, 29)
120 MB


In [16]:
import features

print(features.FILL_CARD_AVG_AMT, features.FILL_AMT_VS_USUAL, features.FILL_MINS_SINCE_LAST)
print(fill_values)
print(features.CATEGORIES == categories)

47.52 1.0 276.0
{'card_avg_amt': np.float64(47.52), 'amt_vs_usual': 1.0, 'mins_since_last': np.float64(276.0)}
True
